**Q3: ESG Message Classification and Triage Using LLMs**

Experiment Overview

This notebook presents a small experiment to explore how Large Language Models (LLMs) can help classify ESG-related operational messages in an organisation.

I tested three employee messages covering water leakage, recycling contamination and accessibility issues. The experiment compares FLAN-T5 Base and Qwen2.5-1.5B-Instruct, followed by a simple rule-based classifier developed in Python.

The main objective is to evaluate the models' ability to produce structured JSON outputs, identify the correct ESG issue categories and recommend appropriate responses.


### 1. Dataset Preparation

In this step, I prepared three sample employee messages related to ESG issues. The messages cover a water leak, recycling contamination and a blocked accessible entrance.

I selected these examples because they represent different operational issues that an organisation may receive from employees. Each message has a unique ID so I can track and compare the classification results.

These three messages will be used to test the LLM and the rule-based classifier to see whether they can identify the correct issue categories and urgency levels.

In [1]:

# Q3: ESG Message Triage
# Step 1: Prepare sample messages

esg_messages = [
    {
        "id": 1,
        "message": "There is a water leak in Building C that has been running all morning."
    },
    {
        "id": 2,
        "message": "The recycling bins are contaminated again and no one seems to be checking them."
    },
    {
        "id": 3,
        "message": "The accessible entrance near the main building has been blocked for two days."
    }
]

for item in esg_messages:
    print(f"Message {item['id']}: {item['message']}")


Message 1: There is a water leak in Building C that has been running all morning.
Message 2: The recycling bins are contaminated again and no one seems to be checking them.
Message 3: The accessible entrance near the main building has been blocked for two days.


### 2. Prompt Design and Improvement

In this step, I improved the original ESG classification prompt to make the instructions clearer and more structured.

I added specific issue categories, urgency levels and recommended teams so the LLM could classify employee messages more consistently. I also included instructions to identify sentiment, follow-up requirements, escalation reasons and data sensitivity risks.

For example, an ongoing water leak should be classified as a water-related issue, while a blocked accessible entrance should be identified as an accessibility issue.

I also instructed the model to return eight fields in JSON format and avoid making assumptions about information that was not provided. This was intended to make the results easier to validate and use in an automated workflow.

### 3. Setting Up the LLM Environment

In this step, I prepared Google Colab to run an open-source language model. I installed the required Python libraries, including Transformers, Accelerate and SentencePiece, which support loading and running language models.

I first selected FLAN-T5 Base to test whether a smaller instruction-following model could classify ESG messages and return the required JSON output.

After loading the model, I checked whether it was running on a CPU or GPU. This helped me confirm that the model was ready for testing.

I used FLAN-T5 Base as my initial model so I could evaluate its performance before trying a stronger model.

In [3]:

# Q3 - Step 4: Install libraries for LLM testing

!pip -q install transformers accelerate sentencepiece



In [4]:

# Q3 - Step 5: Load an open-source LLM

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

print("Model loaded successfully!")
print("Model:", model_name)
print("Running on:", device)



config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded successfully!
Model: google/flan-t5-base
Running on: cpu


### 4. Initial LLM Testing with FLAN-T5 Base

In this step, I tested FLAN-T5 Base using the first employee message about a water leak in Building C.

I used the improved prompt developed earlier and inserted the employee's message into it. The model was then asked to classify the issue and return eight fields in JSON format.

I also added a JSON validation check to see whether the model followed the required output structure.

The model returned only `MEDIUM` instead of a complete JSON response. As a result, the validation failed because the required fields were missing.

This showed that FLAN-T5 Base struggled to follow the structured output instructions, even though it produced an urgency label.

In [5]:

# Q3 - Step 6: Test the first ESG message

import json

# Select the first message
message = esg_messages[0]["message"]

# Insert the message into our improved prompt
prompt = triage_prompt.replace("{message}", message)

# Convert the prompt into model inputs
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(device)

# Generate the model's response
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False
    )

# Decode the response
response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("MESSAGE:")
print(message)

print("\nLLM RESPONSE:")
print(response)

# Check whether the response is valid JSON
try:
    parsed = json.loads(response)
    print("\nJSON VALIDATION: PASS")
except json.JSONDecodeError:
    print("\nJSON VALIDATION: FAIL")


MESSAGE:
There is a water leak in Building C that has been running all morning.

LLM RESPONSE:
MEDIUM

JSON VALIDATION: FAIL


### 5. Testing a Shorter Prompt

After the first test failed to produce valid JSON, I decided to try a shorter prompt to see whether simpler instructions would improve the model's response.

I kept the same water leak message and the eight required JSON fields but removed some of the longer explanations from the original prompt.

However, FLAN-T5 Base returned only `LOW` instead of a complete JSON object. The response failed JSON validation again.

I also noticed that the urgency changed from `MEDIUM` in the first test to `LOW` in the second test, even though the employee's message was the same.

This showed that shortening the prompt did not solve the formatting problem. It also suggested that the model's classification could change depending on how the instructions were written.

Based on these results, I decided to test a stronger instruction-following model, Qwen2.5-1.5B-Instruct.

In [6]:

# Q3 - Step 7: Test a shorter, structured prompt

short_prompt = """
Classify this workplace ESG issue.

Message: There is a water leak in Building C that has been running all morning.

Return a JSON object with exactly these fields:
issue_category, urgency, sentiment, followup_required,
recommended_team, escalation_reason,
data_sensitivity_risk, brief_summary.

Use:
- issue_category: WATER, ENERGY, WASTE, ACCESSIBILITY,
  PROCUREMENT, GOVERNANCE or OTHER
- urgency: LOW, MEDIUM, HIGH or CRITICAL
- sentiment: POSITIVE, NEUTRAL or NEGATIVE
- followup_required: Y or N

Return only valid JSON.
"""

inputs = tokenizer(
    short_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False
    )

short_response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("SHORT PROMPT RESPONSE:")
print(short_response)

try:
    parsed = json.loads(short_response)
    print("\nJSON VALIDATION: PASS")
    print("Fields returned:", len(parsed))
except json.JSONDecodeError:
    print("\nJSON VALIDATION: FAIL")


SHORT PROMPT RESPONSE:
LOW

JSON VALIDATION: FAIL


### 6. Saving and Comparing the Initial Test Results

In this step, I organised the results from my two FLAN-T5 Base tests into a table using Pandas.

The detailed prompt returned `MEDIUM`, while the shorter prompt returned `LOW`. Both responses failed JSON validation because the model did not generate the eight required fields.

By comparing these results, I could see that changing the prompt affected the urgency classification but did not improve the JSON output.

I also saved the results as a CSV file so I could keep a record of the experiment and use the findings in my assessment report.

In [8]:

# Q3 - Step 8: Save initial experiment results

import pandas as pd

initial_results = pd.DataFrame([
    {
        "model": "google/flan-t5-base",
        "prompt_version": "Detailed prompt",
        "message_id": 1,
        "llm_response": response,
        "valid_json": False
    },
    {
        "model": "google/flan-t5-base",
        "prompt_version": "Short prompt",
        "message_id": 1,
        "llm_response": short_response,
        "valid_json": False
    }
])

display(initial_results)

initial_results.to_csv(
    "q3_initial_prompt_tests.csv",
    index=False
)

print("Initial experiment results saved successfully!")


,model,prompt_version,message_id,llm_response,valid_json
0,google/flan-t5-base,Detailed prompt,1,MEDIUM,False
1,google/flan-t5-base,Short prompt,1,LOW,False


Initial experiment results saved successfully!


In [1]:

import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


GPU available: True
GPU name: Tesla T4


In [2]:

# Q3 - Step 10: Load the stronger LLM

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Model loaded successfully!")
print("Model:", model_name)
print("GPU:", torch.cuda.get_device_name(0))


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!
Model: Qwen/Qwen2.5-1.5B-Instruct
GPU: Tesla T4


### 7. Testing Qwen with the First ESG Message

After FLAN-T5 Base failed to generate valid JSON, I decided to test Qwen2.5-1.5B-Instruct to see whether it could follow the classification instructions more accurately.

I started with the same water leak message from Building C so I could compare its performance with the earlier model.

I provided Qwen with the ESG categories, urgency definitions, recommended teams and eight required JSON fields. I also included instructions to avoid making up information.

The model successfully returned valid JSON containing all eight fields. It classified the water leak as `WATER`, assigned `HIGH` urgency and indicated that follow-up was required.

However, I noticed that the model recommended `Facilities, Maintenance` rather than using only the predefined team names. It also described the leak as an immediate safety concern, even though the original message did not clearly mention an immediate danger.

This showed that Qwen was better at following the required JSON format, but its classifications and explanations still needed to be checked.

In [3]:

# Q3 - Step 11: Test Qwen with the first ESG message

import json
import torch

message = "There is a water leak in Building C that has been running all morning."

system_prompt = """
You are an ESG operations triage assistant.

Classify workplace ESG messages using these rules:

Issue categories:
WATER, ENERGY, WASTE, ACCESSIBILITY,
PROCUREMENT, GOVERNANCE, OTHER.

Urgency:
LOW = General enquiry without operational impact.
MEDIUM = Routine issue requiring attention.
HIGH = Ongoing disruption or accessibility barrier.
CRITICAL = Immediate danger to people or the environment.

Sentiment: POSITIVE, NEUTRAL, NEGATIVE.
Follow-up required: Y or N.

Recommended teams:
Facilities, Sustainability, Accessibility,
Procurement, Compliance, Emergency Response,
General Support.

Data sensitivity risk: LOW, MEDIUM, HIGH.

Return exactly one JSON object containing:
issue_category, urgency, sentiment,
followup_required, recommended_team,
escalation_reason, data_sensitivity_risk,
brief_summary.

Do not invent missing details.
Return valid JSON only. No explanation or markdown.
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": message}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

generated_tokens = output[0][inputs["input_ids"].shape[1]:]

response_qwen = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
).strip()

print("MESSAGE:")
print(message)

print("\nQWEN RESPONSE:")
print(response_qwen)

try:
    parsed = json.loads(response_qwen)
    required_fields = [
        "issue_category", "urgency", "sentiment",
        "followup_required", "recommended_team",
        "escalation_reason", "data_sensitivity_risk",
        "brief_summary"
    ]
    missing = [field for field in required_fields if field not in parsed]

    if isinstance(parsed, dict) and not missing:
        print("\nJSON VALIDATION: PASS")
        print("All 8 required fields are present.")
    else:
        print("\nJSON VALIDATION: INCOMPLETE")
        print("Missing fields:", missing)

except json.JSONDecodeError:
    print("\nJSON VALIDATION: FAIL")


MESSAGE:
There is a water leak in Building C that has been running all morning.

QWEN RESPONSE:
{
  "issue_category": "WATER",
  "urgency": "HIGH",
  "sentiment": "NEGATIVE",
  "followup_required": "Y",
  "recommended_team": "Facilities, Maintenance",
  "escalation_reason": "Immediate safety concern due to ongoing leakage.",
  "data_sensitivity_risk": "LOW",
  "brief_summary": "Water leak detected in Building C; urgent action needed."
}

JSON VALIDATION: PASS
All 8 required fields are present.


### 8. Testing Qwen with Three ESG Messages

After successfully testing the first message, I used Qwen2.5-1.5B-Instruct to classify all three ESG messages using the same prompt.

I wanted to see whether the model could identify different types of workplace issues and consistently return the eight required JSON fields.

The model successfully generated valid JSON for all three messages. It correctly identified the water leak as `WATER` and the blocked accessible entrance as `ACCESSIBILITY`.

However, it incorrectly classified the contaminated recycling bins as `ACCESSIBILITY` instead of `WASTE`. It also recommended the Accessibility team for this issue, even though the Sustainability team would have been more appropriate.

This showed that the model could follow the required JSON structure but still make classification errors. I saved the outputs in a CSV file so I could compare them with the rule-based classifier in the next experiment.

In [4]:

# Q3 - Step 12: Test all three ESG messages

import json
import pandas as pd
import torch

test_messages = [
    "There is a water leak in Building C that has been running all morning.",
    "The recycling bins are contaminated again and no one seems to be checking them.",
    "The accessible entrance near the main building has been blocked for two days."
]

qwen_results = []

for i, msg in enumerate(test_messages, start=1):

    chat = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": msg}
    ]

    formatted = tokenizer.apply_chat_template(
        chat,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]

    result = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    try:
        parsed = json.loads(result)
        valid_json = isinstance(parsed, dict)
    except json.JSONDecodeError:
        valid_json = False

    qwen_results.append({
        "message_id": i,
        "message": msg,
        "llm_response": result,
        "valid_json": valid_json
    })

    print(f"\n{'=' * 50}")
    print(f"MESSAGE {i}: {msg}")
    print("\nQWEN RESPONSE:")
    print(result)
    print("\nVALID JSON:", valid_json)

# Save results for assessment evidence
qwen_df = pd.DataFrame(qwen_results)
qwen_df.to_csv("q3_qwen_results.csv", index=False)

print("\nAll Qwen test results saved successfully!")



MESSAGE 1: There is a water leak in Building C that has been running all morning.

QWEN RESPONSE:
{
  "issue_category": "WATER",
  "urgency": "HIGH",
  "sentiment": "NEGATIVE",
  "followup_required": "Y",
  "recommended_team": "Facilities, Maintenance",
  "escalation_reason": "Immediate safety concern due to ongoing leakage.",
  "data_sensitivity_risk": "LOW",
  "brief_summary": "Water leak detected in Building C; urgent action needed."
}

VALID JSON: True

MESSAGE 2: The recycling bins are contaminated again and no one seems to be checking them.

QWEN RESPONSE:
{
  "issue_category": "ACCESSIBILITY",
  "urgency": "HIGH",
  "sentiment": "NEGATIVE",
  "followup_required": "Y",
  "recommended_team": "Accessibility",
  "escalation_reason": "Immediate need for a quick fix to ensure proper waste management.",
  "data_sensitivity_risk": "LOW",
  "brief_summary": "Recycling bins are contaminated, indicating lack of oversight."
}

VALID JSON: True

MESSAGE 3: The accessible entrance near the m

### 9. Developing a Rule-Based Classifier

In this step, I created a simple rule-based classifier using Python to compare its results with Qwen.

Instead of using an LLM, this classifier checks for specific keywords in each employee message and assigns an issue category and responsible team based on predefined rules.

For example, if the message contains words such as "water leak" or "leak", the classifier assigns it to the `WATER` category and recommends the Facilities team. Similarly, messages mentioning "recycling", "waste" or "bins" are classified as `WASTE` and assigned to the Sustainability team.

I also included rules for other issues, such as accessibility and energy-related problems.

The rule-based classifier correctly identified all three test messages. However, I understand that this approach has limitations because it depends on the keywords I included. If an employee describes an issue using different words, the classifier may not recognise it correctly.

I used these results as a baseline to evaluate Qwen's classification accuracy.

In [5]:

# Q3 - Step 13: Rule-based ESG classifier

def rule_based_classifier(message):
    text = message.lower()

    if "water leak" in text or "leak" in text:
        category = "WATER"
        team = "Facilities"

    elif "recycling" in text or "waste" in text or "bins" in text:
        category = "WASTE"
        team = "Sustainability"

    elif "accessible" in text or "accessibility" in text:
        category = "ACCESSIBILITY"
        team = "Accessibility"

    elif "energy" in text or "air conditioning" in text:
        category = "ENERGY"
        team = "Facilities"

    elif "supplier" in text or "procurement" in text:
        category = "PROCUREMENT"
        team = "Procurement"

    elif "policy" in text or "governance" in text:
        category = "GOVERNANCE"
        team = "Compliance"

    else:
        category = "OTHER"
        team = "General Support"

    if "danger" in text or "emergency" in text:
        urgency = "CRITICAL"
    elif (
        "all morning" in text
        or "two days" in text
        or "blocked" in text
    ):
        urgency = "HIGH"
    elif "again" in text or "contaminated" in text:
        urgency = "MEDIUM"
    else:
        urgency = "LOW"

    return {
        "issue_category": category,
        "urgency": urgency,
        "recommended_team": team
    }


# Test the same three messages
baseline_results = []

for i, msg in enumerate(test_messages, start=1):
    result = rule_based_classifier(msg)

    baseline_results.append({
        "message_id": i,
        "message": msg,
        **result
    })

    print(f"\nMessage {i}")
    print(json.dumps(result, indent=2))

baseline_df = pd.DataFrame(baseline_results)

baseline_df.to_csv(
    "q3_rule_based_results.csv",
    index=False
)

print("\nRule-based baseline results saved successfully!")



Message 1
{
  "issue_category": "WATER",
  "urgency": "HIGH",
  "recommended_team": "Facilities"
}

Message 2
{
  "issue_category": "WASTE",
  "urgency": "MEDIUM",
  "recommended_team": "Sustainability"
}

Message 3
{
  "issue_category": "ACCESSIBILITY",
  "urgency": "HIGH",
  "recommended_team": "Accessibility"
}

Rule-based baseline results saved successfully!


### 10. Comparing the LLM and Rule-Based Classifier Results

In this step, I compared Qwen's predictions with the rule-based classifier using the same three ESG messages.

I first identified the expected category for each message and then used Python to check whether each classifier's prediction matched the expected result.

The comparison showed that Qwen correctly classified two out of three messages, achieving 66.67% category accuracy. The rule-based classifier correctly classified all three messages, achieving 100% accuracy.

The main difference was the recycling contamination message. Qwen incorrectly classified it as `ACCESSIBILITY`, while the rule-based classifier correctly identified it as `WASTE`.

Although the rule-based classifier performed better in this experiment, I would not conclude that it is always more accurate. The test only included three messages, and the keyword rules were designed around similar examples.

This comparison helped me understand that an LLM can generate well-structured responses but still make classification errors. Therefore, I would recommend additional validation and human review before using this type of system in a real organisation.

In [6]:

# Q3 - Step 14: Compare LLM and rule-based results

import pandas as pd
import json

expected_categories = ["WATER", "WASTE", "ACCESSIBILITY"]

comparison = []

for i in range(3):
    qwen_output = json.loads(qwen_results[i]["llm_response"])
    baseline_output = baseline_results[i]

    comparison.append({
        "Message": i + 1,
        "Expected": expected_categories[i],
        "Qwen": qwen_output["issue_category"],
        "Rule_Based": baseline_output["issue_category"],
        "Qwen_Correct": qwen_output["issue_category"] == expected_categories[i],
        "Rule_Correct": baseline_output["issue_category"] == expected_categories[i]
    })

comparison_df = pd.DataFrame(comparison)

display(comparison_df)

print("Qwen category accuracy:",
      round(comparison_df["Qwen_Correct"].mean() * 100, 2), "%")

print("Rule-based category accuracy:",
      round(comparison_df["Rule_Correct"].mean() * 100, 2), "%")

comparison_df.to_csv(
    "q3_comparison_results.csv",
    index=False
)

print("Comparison saved successfully!")


,Message,Expected,Qwen,Rule_Based,Qwen_Correct,Rule_Correct
0,1,WATER,WATER,WATER,True,True
1,2,WASTE,ACCESSIBILITY,WASTE,False,True
2,3,ACCESSIBILITY,ACCESSIBILITY,ACCESSIBILITY,True,True


Qwen category accuracy: 66.67 %
Rule-based category accuracy: 100.0 %
Comparison saved successfully!


### 11. Conclusion

Overall, this experiment helped me understand how Large Language Models can be used to classify ESG-related messages and support an organisation's operational processes.

I first tested FLAN-T5 Base using two different prompts, but both tests failed to produce valid JSON. I then tested Qwen2.5-1.5B-Instruct, which successfully generated the required JSON structure for all three messages.

However, Qwen only achieved 66.67% issue-category accuracy because it incorrectly classified the recycling contamination message as an accessibility issue. In comparison, the rule-based classifier achieved 100% category accuracy on the same three messages.

Although the rule-based classifier performed better in this small experiment, it relies on predefined keywords and may struggle with messages written in different ways. Qwen offers more flexibility in understanding language, but its responses still need to be checked for accuracy.

From these findings, I believe that combining an LLM with rule-based validation and human review would be a more reliable approach than depending on the LLM alone.

The experiment was limited to three sample messages, so further testing with a larger and more varied dataset would be necessary before using this system in a real organisation.